In [21]:
import pandas as pd
import numpy as np
import re

DATA_DIR = "../data/transform"

player_stats = pd.read_csv(f"{DATA_DIR}/player_stats.csv")
players = pd.read_csv(f"{DATA_DIR}/players.csv")
matches = pd.read_csv(f"{DATA_DIR}/matches.csv")
clubs = pd.read_csv(f"{DATA_DIR}/clubs.csv")
clubs_per_season = pd.read_csv(f"{DATA_DIR}/clubs_per_season.csv")
squads = pd.read_csv(f"{DATA_DIR}/squads.csv")

dfs = {
    "player_stats": player_stats,
    "players": players,
    "matches": matches,
    "clubs": clubs,
    "clubs_per_season": clubs_per_season,
    "squads": squads,
}

for name, df in dfs.items():
    print("\n", "="*80)
    print(name, df.shape)
    print(df.columns.tolist())
    display(df.head(3))


player_stats (146964, 15)
['player_id', 'match_id', 'club_id', 'goals', 'assists', 'yellow', 'yellow_red', 'red', 'start_eleven', 'minutes', 'on_min', 'off_min', 'team_goals', 'team_conceded', 'rating']


,player_id,match_id,club_id,goals,assists,yellow,yellow_red,red,start_eleven,minutes,on_min,off_min,team_goals,team_conceded,rating
0,284695,3393584,322,0,0,False,False,False,True,90,0,0,2,0,7.5
1,115188,3393584,322,0,0,False,False,False,True,90,0,0,2,0,7.4
2,19279,3393584,322,0,0,False,False,False,True,90,0,0,2,0,7.4



players (5602, 7)
['player_id', 'player_name', 'nationality', 'date_of_birth', 'height', 'position', 'prediction']


,player_id,player_name,nationality,date_of_birth,height,position,prediction
0,883105,Aadil Alleheri,Togo,2005-10-24,NaN,Defensives Mittelfeld,6.755598
1,923834,Aaron Akalé,Frankreich,2005-04-20,1.84,Mittelstürmer,NaN
2,707663,Aaron Appiah,Schweiz,2003-05-18,1.83,Mittelstürmer,NaN



matches (4891, 8)
['match_id', 'season', 'league', 'date', 'home_club_id', 'away_club_id', 'home_goals', 'away_goals']


...

,club_id,club_name,PLZ,location
0,2047,AC Bellinzona,6501,Bellinzona
1,5508,BSC YB U21,3000,Bern
2,1292,Etoile Carouge,1227,Carouge



clubs_per_season (380, 3)
['club_id', 'league', 'season']


,club_id,league,season
0,322,pl,20/21
1,5504,pl,20/21
2,13544,pl,20/21



squads (13030, 3)
['player_id', 'club_id', 'season']


,player_id,club_id,season
0,160275,10724,20/21
1,161794,10724,20/21
2,195913,10724,20/21


In [22]:
POSITION_GROUPS = {
    "goalkeeper": [
        "Torwart"
    ],
    "defense": [
        "Linker Verteidiger",
        "Abwehr",
        "Rechter Verteidiger",
        "Innenverteidiger"
    ],
    "midfield": [
        "Defensives Mittelfeld",
        "Linkes Mittelfeld",
        "Mittelfeld",
        "Offensives Mittelfeld",
        "Zentrales Mittelfeld",
        "Rechtes Mittelfeld"
    ],
    "offense": [
        "Hängende Spitze",
        "Linksaußen",
        "Mittelstürmer",
        "Rechtsaußen",
        "Sturm"
    ]
}

POSITION_TO_GROUP = {
    position: group
    for group, positions in POSITION_GROUPS.items()
    for position in positions
}


def normalize_league(league):
    if pd.isna(league):
        return np.nan

    league = str(league).strip()

    if league == "pl":
        return "Promotion League"

    if league.startswith("1_liga"):
        return "1. Liga"

    return np.nan


def season_to_start_year(season):
    """
    '20/21' -> 2020
    '21/22' -> 2021
    '22/23' -> 2022
    '23/24' -> 2023
    '24/25' -> 2024
    '25/26' -> 2025
    """
    if pd.isna(season):
        return np.nan

    start = str(season).strip().split("/")[0]
    return 2000 + int(start)

In [23]:
df = (
    player_stats
    .merge(
        matches[["match_id", "season", "league", "date"]],
        on="match_id",
        how="left"
    )
    .merge(
        players[["player_id", "player_name", "position"]],
        on="player_id",
        how="left"
    )
)

df["league_group"] = df["league"].apply(normalize_league)
df["season_start_year"] = df["season"].apply(season_to_start_year)
df["position_group"] = df["position"].map(POSITION_TO_GROUP)

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["minutes"] = pd.to_numeric(df["minutes"], errors="coerce")

df = df[
    df["league_group"].isin(["1. Liga", "Promotion League"])
    & df["rating"].notna()
    & df["season_start_year"].notna()
    & df["position_group"].notna()
    & (df["minutes"] > 0)
].copy()

df.head()

,player_id,match_id,club_id,goals,assists,yellow,yellow_red,red,start_eleven,minutes,...,team_conceded,rating,season,league,date,player_name,position,league_group,season_start_year,position_group
0,284695,3393584,322,0,0,False,False,False,True,90,...,0,7.5,20/21,pl,2020-08-15,Kevin Martin,Torwart,Promotion League,2020,goalkeeper
1,115188,3393584,322,0,0,False,False,False,True,90,...,0,7.4,20/21,pl,2020-08-15,Adriano de Pierro,Innenverteidiger,Promotion League,2020,defense
2,19279,3393584,322,0,0,False,False,False,True,90,...,0,7.4,20/21,pl,2020-08-15,Mustafa Sejmenovic,Innenverteidiger,Promotion League,2020,defense
3,126514,3393584,322,0,0,False,False,False,True,90,...,0,7.3,20/21,pl,2020-08-15,William Le Pogam,Linker Verteidiger,Promotion League,2020,defense
4,267582,3393584,322,0,0,False,False,False,True,90,...,0,7.1,20/21,pl,2020-08-15,Axel Danner,Rechter Verteidiger,Promotion League,2020,defense


In [24]:
print(df["league_group"].value_counts())
print(df["position_group"].value_counts())
print(df[["season", "season_start_year"]].drop_duplicates().sort_values("season_start_year"))

league_group
1. Liga             102209
Promotion League     44200
Name: count, dtype: int64
position_group
midfield      50783
defense       49520
offense       36135
goalkeeper     9971
Name: count, dtype: int64
      season  season_start_year
0      20/21               2020
3614   21/22               2021
10813  22/23               2022
19605  23/24               2023
28949  24/25               2024
38314  25/26               2025


In [25]:
season_player_rating = (
    df
    .groupby(
        [
            "player_id",
            "player_name",
            "season",
            "season_start_year",
            "league_group",
            "position",
            "position_group"
        ],
        as_index=False
    )
    .agg(
        avg_rating=("rating", "mean"),
        n_matches=("match_id", "nunique"),
        total_minutes=("minutes", "sum")
    )
)

season_player_rating.head()

,player_id,player_name,season,season_start_year,league_group,position,position_group,avg_rating,n_matches,total_minutes
0,2452,Samel Sabanovic,20/21,2020,1. Liga,Mittelstürmer,offense,6.775000,4,204
1,2452,Samel Sabanovic,20/21,2020,Promotion League,Mittelstürmer,offense,6.500000,2,25
2,2866,Kim Jaggy,20/21,2020,1. Liga,Innenverteidiger,defense,6.800000,10,806
3,2866,Kim Jaggy,21/22,2021,1. Liga,Innenverteidiger,defense,6.989474,19,1090
4,2866,Kim Jaggy,24/25,2024,1. Liga,Innenverteidiger,defense,6.725000,8,284


In [26]:
# Filter for players with minimum matches (at least 5 matches in a season)
season_player_rating_filtered = season_player_rating[
    season_player_rating["n_matches"] >= 5
].copy()

print(f"Filtered from {len(season_player_rating)} to {len(season_player_rating_filtered)} rows")
season_player_rating_filtered.head()

Filtered from 11286 to 8632 rows


,player_id,player_name,season,season_start_year,league_group,position,position_group,avg_rating,n_matches,total_minutes
2,2866,Kim Jaggy,20/21,2020,1. Liga,Innenverteidiger,defense,6.800000,10,806
3,2866,Kim Jaggy,21/22,2021,1. Liga,Innenverteidiger,defense,6.989474,19,1090
4,2866,Kim Jaggy,24/25,2024,1. Liga,Innenverteidiger,defense,6.725000,8,284
6,3391,Markus Neumayr,21/22,2021,1. Liga,Offensives Mittelfeld,midfield,6.900000,9,438
7,4769,Daniel Lopar,21/22,2021,Promotion League,Torwart,goalkeeper,7.010000,10,900


In [27]:
base = season_player_rating_filtered.copy()

one_liga = base[base["league_group"] == "1. Liga"].copy()
promo = base[base["league_group"] == "Promotion League"].copy()

one_liga = one_liga.rename(columns={
    "season": "season_1_liga",
    "season_start_year": "season_year_1_liga",
    "avg_rating": "avg_rating_1_liga",
    "n_matches": "n_matches_1_liga",
    "total_minutes": "minutes_1_liga",
    "position": "position_1_liga",
    "position_group": "position_group_1_liga"
})

promo = promo.rename(columns={
    "season": "season_promotion",
    "season_start_year": "season_year_promotion",
    "avg_rating": "avg_rating_promotion",
    "n_matches": "n_matches_promotion",
    "total_minutes": "minutes_promotion",
    "position": "position_promotion",
    "position_group": "position_group_promotion"
})

pairs = one_liga.merge(
    promo,
    on=["player_id", "player_name"],
    how="inner"
)

pairs = pairs[
    (pairs["season_year_promotion"] - pairs["season_year_1_liga"]).abs() == 1
].copy()

pairs["rating_diff_promotion_minus_1_liga"] = (
    pairs["avg_rating_promotion"] - pairs["avg_rating_1_liga"]
)

pairs["transition_direction"] = np.where(
    pairs["season_year_promotion"] > pairs["season_year_1_liga"],
    "1. Liga -> Promotion League",
    "Promotion League -> 1. Liga"
)

pairs["same_position_group"] = (
    pairs["position_group_1_liga"] == pairs["position_group_promotion"]
)

pairs.head()

,player_id,player_name,season_1_liga,season_year_1_liga,league_group_x,position_1_liga,position_group_1_liga,avg_rating_1_liga,n_matches_1_liga,minutes_1_liga,...,season_year_promotion,league_group_y,position_promotion,position_group_promotion,avg_rating_promotion,n_matches_promotion,minutes_promotion,rating_diff_promotion_minus_1_liga,transition_direction,same_position_group
0,33347,Daniele Russo,21/22,2021,1. Liga,Innenverteidiger,defense,6.933333,18,1483,...,2020,Promotion League,Innenverteidiger,defense,6.784615,13,1086,-0.148718,Promotion League -> 1. Liga,True
6,33771,Patrick Rossini,22/23,2022,1. Liga,Mittelstürmer,offense,7.203571,28,2273,...,2021,Promotion League,Mittelstürmer,offense,6.922222,18,1144,-0.281349,Promotion League -> 1. Liga,True
7,44661,Stefan Glarner,21/22,2021,1. Liga,Innenverteidiger,defense,6.922727,22,1949,...,2020,Promotion League,Innenverteidiger,defense,6.800000,5,448,-0.122727,Promotion League -> 1. Liga,True
12,45023,Christian Leite,23/24,2023,1. Liga,Torwart,goalkeeper,7.242857,14,1260,...,2022,Promotion League,Torwart,goalkeeper,7.115000,20,1664,-0.127857,Promotion League -> 1. Liga,True
16,45173,Ilker Tugal,21/22,2021,1. Liga,Innenverteidiger,defense,6.686667,15,669,...,2020,Promotion League,Innenverteidiger,defense,6.910000,10,900,0.223333,Promotion League -> 1. Liga,True


In [28]:
print("Anzahl Spieler-Saison-Paare:", len(pairs))
print(pairs["transition_direction"].value_counts())
print(pairs["same_position_group"].value_counts())

Anzahl Spieler-Saison-Paare: 765
transition_direction
1. Liga -> Promotion League    403
Promotion League -> 1. Liga    362
Name: count, dtype: int64
same_position_group
True    765
Name: count, dtype: int64


In [29]:
summary_by_position = (
    pairs
    .groupby("position_group_1_liga", as_index=False)
    .agg(
        mean_diff=("rating_diff_promotion_minus_1_liga", "mean"),
        median_diff=("rating_diff_promotion_minus_1_liga", "median"),
        std_diff=("rating_diff_promotion_minus_1_liga", "std"),
        n_pairs=("rating_diff_promotion_minus_1_liga", "count"),
        mean_rating_1_liga=("avg_rating_1_liga", "mean"),
        mean_rating_promotion=("avg_rating_promotion", "mean"),
        mean_matches_1_liga=("n_matches_1_liga", "mean"),
        mean_matches_promotion=("n_matches_promotion", "mean"),
        mean_minutes_1_liga=("minutes_1_liga", "mean"),
        mean_minutes_promotion=("minutes_promotion", "mean")
    )
    .sort_values("mean_diff", ascending=False)
)

summary_by_position

,position_group_1_liga,mean_diff,median_diff,std_diff,n_pairs,mean_rating_1_liga,mean_rating_promotion,mean_matches_1_liga,mean_matches_promotion,mean_minutes_1_liga,mean_minutes_promotion
1,goalkeeper,-0.092553,-0.112179,0.160430,49,7.187188,7.094635,18.897959,19.081633,1678.367347,1698.551020
0,defense,-0.110437,-0.123333,0.195836,227,6.882004,6.771567,17.753304,17.312775,1343.431718,1204.026432
2,midfield,-0.143551,-0.133201,0.193250,244,6.941038,6.797487,18.459016,17.766393,1251.053279,1030.565574
3,offense,-0.199260,-0.204620,0.294100,245,6.955125,6.755866,18.363265,17.163265,1213.942857,878.681633


In [30]:
stable_pairs = pairs[pairs["same_position_group"]].copy()

summary_stable_positions = (
    stable_pairs
    .groupby("position_group_1_liga", as_index=False)
    .agg(
        mean_diff=("rating_diff_promotion_minus_1_liga", "mean"),
        median_diff=("rating_diff_promotion_minus_1_liga", "median"),
        std_diff=("rating_diff_promotion_minus_1_liga", "std"),
        n_pairs=("rating_diff_promotion_minus_1_liga", "count"),
        mean_rating_1_liga=("avg_rating_1_liga", "mean"),
        mean_rating_promotion=("avg_rating_promotion", "mean")
    )
    .sort_values("mean_diff", ascending=False)
)

summary_stable_positions

,position_group_1_liga,mean_diff,median_diff,std_diff,n_pairs,mean_rating_1_liga,mean_rating_promotion
1,goalkeeper,-0.092553,-0.112179,0.160430,49,7.187188,7.094635
0,defense,-0.110437,-0.123333,0.195836,227,6.882004,6.771567
2,midfield,-0.143551,-0.133201,0.193250,244,6.941038,6.797487
3,offense,-0.199260,-0.204620,0.294100,245,6.955125,6.755866


In [33]:
print(
    summary_by_position[["position_group_1_liga", "median_diff"]]
    .assign(median_diff=lambda x: x["median_diff"].round(2))
    .sort_values("position_group_1_liga")
    .to_string(index=False)
)

position_group_1_liga  median_diff
              defense        -0.12
           goalkeeper        -0.11
             midfield        -0.13
              offense        -0.20
